In [21]:
from dotenv import load_dotenv

load_dotenv()

True

In [1]:
### dpo 데이터셋 형식 맞추기

import json

def process_jsonl_file(input_file, output_file, start_index, end_index):
    # 데이터를 저장할 리스트 초기화
    extracted_data = []

    # jsonl 파일을 열고 데이터 추출
    with open(input_file, 'r', encoding='utf-8') as infile:
        # 파일을 한 줄씩 읽으면서 처리
        for i, line in enumerate(infile):
            if i < start_index:
                continue  # 시작 인덱스 전의 데이터는 무시
            if i >= end_index:
                break  # 종료 인덱스 이후의 데이터는 무시

            # JSON 데이터를 파싱
            record = json.loads(line)
            text = record.get('text', '')

            # 'text' 값을 "Assistant: " 기준으로 나누기
            if 'Assistant: ' in text:
                part1, part2 = text.split('Assistant: ', 1)
                part1 += "<output_format>\n{\n\t\"same_form_judgment\": \"\",\n\t\"summary\": \"\"\n}\n</output_format>\n" + 'Assistant: '  # part1에 "Assistant: " 포함
                
                # 새로운 레코드 구성
                new_record = {
                    'question': part1,
                    'response_j': part2,
                    'response_k': ''
                }
                extracted_data.append(new_record)

    # 결과를 새로운 파일에 저장
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for record in extracted_data:
            json.dump(record, outfile, ensure_ascii=False)
            outfile.write('\n')

# 사용 예
input_file = '/home/ragllama/decs_jupyter_lab/korean-finetuning/Suchae/langserve/DPO_dataset/primary_dataset_nk_ngpt.jsonl'
output_file = '/home/ragllama/decs_jupyter_lab/korean-finetuning/Suchae/langserve/DPO_dataset/judgment_transducer_DPO_dataset.jsonl'
start_index = 128000
end_index = 158000

process_jsonl_file(input_file, output_file, start_index, end_index)


In [2]:
### 답변 json 형식이 이상해서 수정


import re
import json

def process_jsonl(input_file, output_file):
    # 정규 표현식 패턴
    pattern = r'(?<="same_form_judgment":\s)(.*?)(?=,\s+"summary": )|(?<="summary": )(.*?)(?=\n\})'

    # 입력 파일을 열고 읽기
    with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
        for line in infile:
            # JSON 문자열을 파싱
            try:
                json_obj = json.loads(line.strip())
            except json.JSONDecodeError as e:
                print(f"JSON 오류: {e} - 원본 데이터: {line.strip()}")
                continue
            # 'response_j' 필드가 존재하는지 확인
            if 'response_j' in json_obj:
                response_j = json_obj['response_j']

                # 정규 표현식으로 추출
                matches = re.findall(pattern, response_j, re.DOTALL)

                # 결과를 저장할 변수
                modified_response_j = response_j

                # 정규 표현식으로 추출한 값들로부터 문자열 수정
                for match in matches:
                    for i, group in enumerate(match):
                        if group:  # 빈 문자열이 아닌 경우만
                            if i == 0:
                                modified_response_j = modified_response_j.replace(match[0], f'"{group.strip()}"', 1)
                            else:
                                modified_response_j = modified_response_j.replace(match[1], f'"{group.strip()}"', 1)

                # 수정된 'response_j' 필드를 원래 JSON 객체에 적용
                json_obj['response_j'] = modified_response_j

            # 수정된 JSON 객체를 파일에 저장
            outfile.write(json.dumps(json_obj, ensure_ascii=False) + '\n')

# 파일 경로 설정
input_file = 'judgment_transducer_DPO_dataset.jsonl'
output_file = 'judgment_transducer_DPO_dataset_2.jsonl'

# 처리 함수 호출
process_jsonl(input_file, output_file)
